# Proving what first-party signals are worth, without fooling ourselves

One notebook, two kinds of data:

* **PUBLIC DATA**: the Criteo Uplift Prediction Dataset v2.1, a real randomised ad-incrementality test
  (~14 million users, CC BY-NC-SA 4.0). Only aggregates are used here (`sql/01_criteo_lift.sql`).
* **SIMULATED**: six fictional clients of a first-party signal platform (`src/simulate.py`, seed 42), with eight
  measurement traps planted. Not Datahash's data, not any client's data.

Every number of the report, the dashboard, the workbook and the client deliverables comes from `src/analysis.py`,
which this notebook calls and explains.

In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import analysis as A
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
OUT, PROC = ROOT / "data" / "output", ROOT / "data" / "processed"
H = json.loads((OUT / "headline.json").read_text(encoding="utf-8"))

## 1. Exposed is not incremental (Criteo, PUBLIC DATA)

Users were randomly split: 85% *eligible* to see the ads (treatment), 15% held out (control). Only a small share of
the eligible users were actually reached (`exposure`).

**Intention to treat (ITT)**: compare the two randomised groups as a whole. Randomisation makes them identical except
for the ads, so the difference is caused by the ads. Interval: normal approximation for two proportions; relative
lift interval by the delta method on log(p1/p0).

In [2]:
g = pd.read_parquet(PROC / "criteo_by_group.parquet").set_index("treatment")
n0, n1 = g.loc[0, "users"], g.loc[1, "users"]
p0, p1 = g.loc[0, "conversion_rate"], g.loc[1, "conversion_rate"]
se = np.sqrt(p0 * (1 - p0) / n0 + p1 * (1 - p1) / n1)
se_log = np.sqrt((1 - p0) / (n0 * p0) + (1 - p1) / (n1 * p1))
print(f"control {p0:.4%} vs treatment {p1:.4%}  ->  ITT +{p1 - p0:.4%} points")
print(f"relative lift {p1 / p0 - 1:+.1%}  (95% {np.exp(np.log(p1 / p0) - 1.96 * se_log) - 1:+.1%} to {np.exp(np.log(p1 / p0) + 1.96 * se_log) - 1:+.1%})")

control 0.1938% vs treatment 0.3089%  ->  ITT +0.1152% points
relative lift +59.4%  (95% +54.4% to +64.7%)


**Effect on the exposed (Wald estimator / CACE).** Nobody in the control group can be exposed, so the whole ITT effect
comes from the reached users: effect per reached user = ITT / exposure rate. This is still a randomised estimate.

**The trap**: comparing exposed with unexposed users inside the treatment group. The ad system chose whom to reach,
and it reached users who were already likely to convert.

In [3]:
c = H["criteo"]
print(f"exposure rate in treatment: {c['exposure_rate']:.2%}")
print(f"effect per exposed user: +{c['cace']:.2%} points (95% {c['cace_lo']:.2%} to {c['cace_hi']:.2%})")
print(f"exposed users convert at {c['cr_exposed']:.2%}; without ads they would have converted at {c['counterfactual_exposed']:.2%}"
      f" -> +{c['cace_rel']:.0%} relative")
print(f"naive exposed vs unexposed ratio: x{c['naive_ratio']:.0f} (+{c['naive_rel']:.0%}), i.e. ~{c['naive_rel'] / c['cace_rel']:.0f} times the true effect")
print(f"share of exposed users' conversions caused by the ads: {c['incremental_share_exposed']:.0%}")
pd.read_parquet(PROC / "criteo_by_propensity.parquet").pivot(index="decile", columns="treatment", values=["exposure_rate", "conversion_rate"])

exposure rate in treatment: 3.60%
effect per exposed user: +3.20% points (95% 3.01% to 3.38%)
exposed users convert at 5.38%; without ads they would have converted at 2.18% -> +146% relative
naive exposed vs unexposed ratio: x45 (+4403%), i.e. ~30 times the true effect
share of exposed users' conversions caused by the ads: 59%


exposure_rate        conversion_rate       
treatment             0      1               0      1
decile                                               
1                0.0000 0.0160          0.0003 0.0004
2                0.0000 0.0204          0.0002 0.0003
3                0.0000 0.0163          0.0002 0.0004
4                0.0000 0.0200          0.0002 0.0003
5                0.0000 0.0168          0.0002 0.0004
6                0.0000 0.0207          0.0001 0.0004
7                0.0000 0.0171          0.0002 0.0004
8                0.0000 0.0220          0.0003 0.0004
9                0.0000 0.0603          0.0019 0.0030
10               0.0000 0.1492          0.0166 0.0247

Selection is visible in the deciles of a pre-treatment feature: the top two deciles hold most of the exposure **and**
the highest conversion rates *in the control group*, where nobody saw an ad.

**Power.** How many users does a lift study need? Two-proportion z-test, 5% two-sided, 80% power, 85/15 split.

In [4]:
pd.read_csv(OUT / "criteo_power.csv")

,relative_lift,control_users_85_15_split,users_per_arm_50_50
0,0.0200,11926785,20416759
1,0.0500,1916830,3315097
2,0.1000,482767,848945
3,0.2000,122471,222319
4,0.5900,14870,30061


## 2. The simulated panel: naive reads vs controlled reads (SIMULATED)

`sql/02_pre_post.sql` computes, for each use case: the naive before/after change in the treated market, the change
relative to the control market (difference-in-differences on ratios), and the same weeks last year. Baseline = 8
weeks before go-live; ramp-up excluded; read window fixed in `use_cases`.

In [5]:
pd.read_csv(OUT / "sql_02_pre_post.csv")

,use_case_id,reported_naive,business_naive,reported_vs_control,business_vs_control,business_same_weeks_last_year,business_vs_last_year
0,UC1,0.1091,-0.0177,0.1685,0.0308,-0.0641,0.0496
1,UC2,0.4363,0.0735,0.3287,-0.0059,NaN,NaN
2,UC3,0.1830,0.0715,0.1162,-0.0066,NaN,NaN
3,UC4,0.2368,0.1790,0.0892,0.0295,NaN,NaN
4,UC5,0.0629,0.0178,0.0440,0.0032,0.0184,-0.0006
5,UC7,0.0831,0.0331,0.0844,0.0427,-0.0183,0.0523


## 3. Causal impact (structural time series), explained simply

1. Treated and control markets share the calendar (Ramadan, Eid, sales), so we model the **ratio** treated / control,
   by week, in logs. Holidays that hit only one of the two markets (national days) enter as a regressor.
2. Before go-live the ratio follows a slowly drifting **local level** (`statsmodels` `UnobservedComponents`).
3. We project the ratio over the read window 1,000 times with its uncertainty and multiply by the control: 1,000
   plausible "no change" paths. Effect = actual / counterfactual - 1; 95% interval = 2.5-97.5 percentiles.
4. **In-time placebos**: the same read at fake go-live dates in the pre-period (true effect = 0) shows how large an
   effect noise alone produces. Rule: retained only if the interval excludes 0 **and** placebo p < 0.05.
5. If more than 25% of the read window is a seasonal peak, the effect must also hold in the 8 following weeks.

In [6]:
uc = pd.read_parquet(PROC / "use_cases.parquet").set_index("use_case_id")
r = uc.loc["UC1"]
biz = pd.read_parquet(PROC / "business_daily.parquet")
y = A.weekly(A.series(biz, "C1", "KSA"), r.go_live)
x = A.weekly(A.series(biz, "C1", "UAE"), r.go_live)
pre_b, post_b = A.windows(r)
ci, path = A.causal_impact(y, x, pre_b, post_b, X=A.calendar_exog("KSA", "UAE", r.go_live))
print({k: round(v, 4) for k, v in ci.items()})
path.tail(8)

{'effect': 0.0322, 'lo': 0.0166, 'hi': 0.0465, 'p_positive': 1.0, 'placebo_n': 23, 'placebo_sd': 0.0072, 'placebo_p': 0.0}


,block,actual,counterfactual,cf_lo,cf_hi
51,-1,13933,"13,717.5380",NaN,NaN
52,2,15938,"15,085.6501","14,581.6915","15,627.0169"
53,3,15337,"14,728.1361","14,240.2097","15,219.5609"
54,4,14862,"14,466.0030","13,922.0706","14,988.8664"
55,5,14624,"14,584.6744","14,085.5639","15,075.5783"
56,6,16615,"16,448.4841","15,867.5038","17,030.8754"
57,7,15789,"15,257.7873","14,749.8336","15,784.9059"
58,8,16180,"15,386.2367","14,901.2397","15,927.0075"


In [7]:
reads = pd.read_csv(OUT / "use_case_reads.csv")
reads[["use_case_id", "client_label", "use_case", "platform", "reported_naive", "reported_effect", "reported_lo", "reported_hi",
       "business_naive", "business_effect", "business_lo", "business_hi", "business_placebo_p", "season_overlap", "verdict"]]

,use_case_id,client_label,use_case,platform,reported_naive,reported_effect,reported_lo,reported_hi,business_naive,business_effect,business_lo,business_hi,business_placebo_p,season_overlap,verdict
0,UC1,Client A · Retail,Web CAPI,Meta,0.1091,0.1837,0.1335,0.2292,-0.0177,0.0322,0.0166,0.0465,0.0000,0.0816,Incremental business effect
1,UC2,Client B · Quick-service restaurant,Web CAPI,TikTok,0.4363,0.3291,0.2938,0.3652,0.0735,-0.0032,-0.0174,0.0113,0.8750,0.0000,Not real: duplicates (event_id missing)
2,UC3,Client B · Quick-service restaurant,Web CAPI,Snapchat,0.1830,0.1066,0.0766,0.1384,0.0715,-0.0040,-0.0160,0.0085,0.8333,0.0000,"Signal recovered, no business effect shown"
3,UC4,Client C · Telecom,CRM Conversions API,Meta,0.2368,0.0781,0.0351,0.1217,0.1790,0.0268,0.0077,0.0454,0.0455,0.6071,Seasonal: naive uplift not retained
4,UC5,Client D · Streaming,App CAPI,Meta,0.0629,0.0297,0.0055,0.0551,0.0178,-0.0042,-0.0152,0.0075,0.3478,0.0000,"Signal recovered, no business effect shown"
5,UC7,Client E · Bank & insurance,CRM Conversions API,Meta,0.0831,0.0739,0.0108,0.1394,0.0331,0.0537,0.0342,0.0735,0.0000,0.0714,Incremental business effect


## 4. Breaking a great-looking uplift (SIMULATED)

**Broken deduplication** (`sql/03_dedup_check.sql`): server events without `event_id` cannot be matched to the pixel
copy, so each purchase counts twice.

In [8]:
t = H["traps"]
pd.DataFrame({"dedup": t["dedup"]}).T

,flag_weeks,first_flag,event_id_coverage_broken,event_id_coverage_fixed,reported_per_order_before,reported_per_order_broken,reported_per_order_after_fix,inflation_vs_before
dedup,8,2025-09-01,0.0000,0.9700,0.1121,0.1506,0.1128,0.3435


**Ramp-up**: the first week after go-live (learning phase) vs the steady weeks. Including it dilutes the read.

In [9]:
{k: (round(v["effect"], 4) if isinstance(v, dict) else v) for k, v in t["ramp"].items()}

{'with_ramp_4w': 0.1106,
 'without_ramp_4w': 0.1899,
 'ramp_days_effect': -0.0782,
 'ramp_days': 10}

**Attribution window change** (`sql/04_attribution_changes.sql`): the account moved from 7-day click + 1-day view to
7-day click. The reported series breaks; the 7-day click column (unchanged) and client outcomes do not.

**Click date vs conversion date**: by click date, recent days are still filling up.

**EMQ**: signal health improved sharply with no effect on subscriptions. **GA4 server-side**: changes in GA4 counts,
read relative to client orders.

In [10]:
pd.DataFrame({k: pd.Series(t[k]) for k in ("window", "lag", "emq", "season")})

,window,lag,emq,season
business_change,-0.0691,NaN,NaN,NaN
business_effect,NaN,NaN,-0.0042,0.0268
business_hi,NaN,NaN,0.0075,0.0454
business_lo,NaN,NaN,-0.0152,0.0077
business_naive,NaN,NaN,NaN,0.1790
changed_on,2026-06-14,NaN,NaN,NaN
click_date_change_last3w,NaN,-0.0685,NaN,NaN
consistent_7d_click_change,-0.0891,NaN,NaN,NaN
conv_date_change_last3w,NaN,-0.0121,NaN,NaN
emq_after,NaN,NaN,8.2786,NaN


In [11]:
pd.Series(t["ga4"])

sessions_change                     0.0109
new_users_change                   -0.1706
returning_users_change              0.2039
new_share_before                    0.5127
new_share_after                     0.4202
unassigned_before                   0.0850
unassigned_after                    0.0401
ga4_purchases_change               -0.0082
business_orders_change             -0.0723
sessions_per_order_change           0.0897
new_users_per_order_change         -0.1059
returning_users_per_order_change    0.2978
ga4_purchases_per_order_change      0.0691
dtype: float64

## 5. Geo test: synthetic control and placebos (SIMULATED client F)

Six treated regions bid on the Google tag gateway conversion action; 18 control regions keep the old one.

* **Synthetic control**: non-negative weights summing to 1 on the control regions, chosen so the weighted mix tracks
  the treated regions over 78 pre-weeks (`scipy.optimize`, SLSQP). Effect = treated / synthetic - 1 over 7 test weeks.
* **Placebo groups**: 300 fake treated groups of 6 control regions, each with its own synthetic control from the other
  12. Their effects show the noise; the 95% interval is the estimate minus the placebo quantiles.
* **In-time placebo**: the same method on a fake go-live 20 weeks earlier.
* **Power, planned before the test**: minimum detectable effect from the pre-period noise of the gap.

In [12]:
geo = H["geo"]
{k: geo[k] for k in ("effect", "lo", "hi", "p_value", "placebo_in_time", "naive_prepost", "did", "rmspe_pre", "mde_by_weeks", "top_donors")}

{'effect': 0.049129810559574105,
 'lo': 0.028845252696007424,
 'hi': 0.07516353029683083,
 'p_value': 0.0,
 'placebo_in_time': -0.010301117279025052,
 'naive_prepost': 0.15825529786439252,
 'did': 0.08068810342803534,
 'rmspe_pre': 0.01617192638011859,
 'mde_by_weeks': {'4': 0.028594714671391346,
  '7': 0.021615572523243295,
  '10': 0.01808488548084597,
  '13': 0.015861493839225218},
 'top_donors': {'Dubai': 0.196, 'Riyadh': 0.18, 'Pune': 0.111}}

In [13]:
pd.read_csv(OUT / "geo_weights.csv")

,Unnamed: 0,weight
0,Dubai,0.1963
1,Riyadh,0.1798
2,Pune,0.1115
3,Mumbai,0.0832
4,Makkah,0.0736
5,Kolkata,0.0673
6,Abha,0.0517
7,Jaipur,0.0488
8,Ras Al Khaimah,0.0439
9,Ajman,0.0365


## 6. Detection score

Each planted trap is checked by a rule that does not look at the truth; the hidden truth file (used once, deleted
before publication) then says whether the read was right and whether the 95% interval covered the programmed effect.

In [14]:
pd.read_csv(OUT / "detection.csv")

,event,check,detected,trap,truth,true_reported_change,true_business_change,interval_covers_truth
0,1,"True uplift kept, sized on business not on pla...",True,True uplift,"Meta-reported conversions +18%, client revenue...",0.1800,0.0400,True
1,2,"Broken dedup flagged, uplift rejected",True,Broken deduplication,Server events sent without event_id until 26 O...,0.3500,0.0000,True
2,3,Ramp-up excluded (learning dip visible),True,Ramp-up,10-day learning phase after go-live (dip then ...,NaN,NaN,NaN
3,4,Seasonal false signal not retained,True,Seasonal false signal,Read window falls on Ramadan 2026: naive pre/p...,0.0600,0.0000,False
4,5,"Window change found, consistent read used",True,Attribution window change,Meta setting 7d click + 1d view -> 7d click on...,0.1000,0.0600,True
5,6,EMQ gain not sold as value,True,"EMQ up, no value",EMQ 4.6 -> 8.3 on App CAPI with no business ef...,0.0500,0.0000,True
6,7,Geo effect found with interval,True,Geo test,Google Tag Gateway as bidding signal in 6 trea...,0.1100,0.0500,True
7,8,Click-date dip not read as a drop,True,Click-date lag,Conversions by click date incomplete for the l...,NaN,NaN,NaN


Reading: all eight traps are caught. One interval misses its truth: the CRM read of client C during Ramadan
(a small positive residual, true effect 0), which is why the seasonal persistence rule exists. Its persistence read did not hold, so the
uplift was not retained. A 95% interval is expected to miss about one time in twenty.